# Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import json
import time
import clifpy
from clifpy import compute_sofa_polars
#from clifpy.clif_orchestrator import ClifOrchestrator
from clifpy.tables.respiratory_support import RespiratorySupport
from clifpy.utils.comorbidity import calculate_elix
from clifpy.utils.stitching_encounters import stitch_encounters
from clifpy.utils.sofa_polars import _load_and_convert_medications
import polars as pl
import duckdb

# Global Settings

In [ ]:
pd.set_option('display.max_columns', None)

os.makedirs('output_no_share', exist_ok=True)
os.makedirs('output_to_box', exist_ok=True)

con = duckdb.connect(database='output_no_share/cardiac_hospitalization_id.duckdb')

with open("config.json", "r", encoding="utf-8-sig") as f:
    cfg = json.load(f)

clif_path = cfg["data_directory"]
print(f"CLIF filepath: {clif_path}")

file_type = cfg['filetype']
print(f'Filetype: {file_type}')

time_zone = cfg['timezone']
print(f'Timezone: {time_zone}')

site_name = cfg.get('site_name', 'site')
print(f'Site name: {site_name}')

con.execute(f"SET TimeZone = '{time_zone}'")

File path: Z:/DataStageData/CQODE DB Backbone/stable_2026_01/rclif
Time zone: America/Chicago


# File paths

In [ ]:
ext = file_type   # "parquet" or "csv"
hosp_diagnosis_path    = f"{clif_path}/clif_hospital_diagnosis.{ext}"
procedure_path         = f"{clif_path}/clif_patient_procedures.{ext}"
hospitalization_path   = f"{clif_path}/clif_hospitalization.{ext}"
dnr_path               = f"{clif_path}/clif_code_status.{ext}"
patient_path           = f"{clif_path}/clif_patient.{ext}"
vent_path              = f"{clif_path}/clif_respiratory_support.{ext}"
dialysis_path          = f"{clif_path}/clif_crrt_therapy.{ext}"
patient_diagnosis_path = f"{clif_path}/clif_patient_diagnosis.{ext}"
adt_path               = f"{clif_path}/clif_adt.{ext}"
vitals_path            = f"{clif_path}/clif_vitals.{ext}"
assessment_path        = f"{clif_path}/clif_patient_assessments.{ext}"
micro_culture_path     = f"{clif_path}/clif_microbiology_culture.{ext}"
labs_path              = f"{clif_path}/clif_labs.{ext}"
intermittent_med_path  = f"{clif_path}/clif_medication_admin_intermittent.{ext}"
continuous_med_path    = f"{clif_path}/clif_medication_admin_continuous.{ext}"

In [ ]:
REQUIRED = {
    hosp_diagnosis_path:  ["hospitalization_id", "diagnosis_code", "poa_present", "diagnosis_primary"],
    procedure_path:       ["hospitalization_id", "procedure_code", "procedure_billed_dttm"],
    hospitalization_path: ["hospitalization_id", "patient_id", "admission_dttm", "discharge_dttm",
                           "age_at_admission", "admission_type_name", "admission_type_category",
                           "discharge_category"],
    adt_path:             ["hospitalization_id", "in_dttm", "out_dttm", "location_category"],
    patient_path:         ["patient_id", "sex_category", "race_category", "ethnicity_category",
                           "language_category", "death_dttm"],
    assessment_path:      ["hospitalization_id", "recorded_dttm", "assessment_category",
                           "numerical_value"],
}

errors = []
for path, required_cols in REQUIRED.items():
    name = os.path.basename(path)
    if not os.path.exists(path):
        errors.append(f"  ✗ MISSING FILE:  {name}")
        continue
    actual_cols = set(duckdb.sql(f"SELECT * FROM '{path}' LIMIT 0").columns)
    missing = [c for c in required_cols if c not in actual_cols]
    if missing:
        errors.append(f"  ✗ MISSING COLS:  {name} → {missing}")
    else:
        print(f"  ✓ {name}")

if errors:
    print("\nPRE-FLIGHT FAILED:")
    for e in errors:
        print(e)
    raise RuntimeError("Fix the above issues before running the notebook.")
else:
    print("\nAll required CLIF tables present and columns verified. Ready to run.")